# Feature importance with decision trees and random forests
1. A random forest's feature importance on MNIST: which pixels matter.
2. How a decision tree computes feature importance, by hand, on 5 and 15 rows.
3. How a random forest combines its trees' importances.
4. The bias towards high-cardinality columns, and permutation importance.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.datasets import fetch_openml, make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

## 1. MNIST: 42,000 images of 28 x 28 = 784 pixels
`fetch_openml` downloads the full MNIST set once (about 15 MB) and caches it; we keep a random 42,000 images, the
size of the Kaggle Digit Recognizer file.

In [ ]:
X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)
idx = np.random.default_rng(0).choice(len(X), 42000, replace=False)
X, y = X[idx], y[idx]
print(X.shape)

In [ ]:
# train on all 42,000 images: we only want the importances, not a test score
rf = RandomForestClassifier(random_state=42, n_jobs=-1).fit(X, y)
imp = rf.feature_importances_
print(imp.shape, "sum:", imp.sum().round(6))
print("pixels with zero importance:", int((imp == 0).sum()))

In [ ]:
# one number per pixel, so reshape back to the 28 x 28 image grid
px.imshow(imp.reshape(28, 28), color_continuous_scale="Blues", title="feature importance of each pixel")

## 2. Feature importance in one decision tree, by hand: 5 rows, 2 columns

In [ ]:
X5, y5 = make_classification(n_samples=5, n_classes=2, n_features=2, n_informative=2, n_redundant=0, random_state=0)
tree = DecisionTreeClassifier(random_state=0).fit(X5, y5)
print(export_text(tree, show_weights=True, decimals=3))
print("feature_importances_:", tree.feature_importances_)

In [ ]:
def node_decreases(tree):
    """Weighted impurity decrease of every split node: N_t/N * (G_t - N_L/N_t * G_L - N_R/N_t * G_R)."""
    t = tree.tree_
    N = t.n_node_samples[0]
    rows = []
    for i in range(t.node_count):
        L, R = t.children_left[i], t.children_right[i]
        if L == -1:                                   # a leaf: no split, no decrease
            continue
        n_t, n_l, n_r = t.n_node_samples[i], t.n_node_samples[L], t.n_node_samples[R]
        delta = n_t / N * (t.impurity[i] - n_l / n_t * t.impurity[L] - n_r / n_t * t.impurity[R])
        rows.append({"node": i, "column": t.feature[i], "rows": n_t, "gini": round(t.impurity[i], 4),
                     "decrease": round(delta, 4)})
    return pd.DataFrame(rows)


d = node_decreases(tree)
print(d)
by_column = d.groupby("column")["decrease"].sum()
print("importance by hand:", (by_column / by_column.sum()).round(4).to_dict())

## 3. The same with 15 rows: column 0 now splits two nodes

In [ ]:
X15, y15 = make_classification(n_samples=15, n_classes=2, n_features=2, n_informative=2, n_redundant=0, random_state=0)
tree15 = DecisionTreeClassifier(random_state=0).fit(X15, y15)
print(export_text(tree15, show_weights=True, decimals=3))
d = node_decreases(tree15)
print(d)
by_column = d.groupby("column")["decrease"].sum()
print("importance by hand:", (by_column / by_column.sum()).round(4).to_dict())
print("feature_importances_:", tree15.feature_importances_.round(4))

## 4. A random forest averages its trees' importances

In [ ]:
rf2 = RandomForestClassifier(n_estimators=2, random_state=3).fit(X5, y5)
for k, t in enumerate(rf2.estimators_, start=1):
    print(f"tree {k}:", t.feature_importances_.round(4))
print("mean of the trees:", np.mean([t.feature_importances_ for t in rf2.estimators_], axis=0).round(4))
print("forest:          ", rf2.feature_importances_.round(4))

## 5. High cardinality: a useless column with many different values
Four real columns, plus `random_id` (1,000 different values) and `random_coin` (2 values). Neither has anything to do
with the target.

In [ ]:
Xc, yc = make_classification(n_samples=1000, n_features=4, n_informative=3, n_redundant=1, flip_y=0.1, random_state=0)
df = pd.DataFrame(Xc.round(1), columns=["x1", "x2", "x3", "x4"])
rng = np.random.default_rng(0)
df["random_id"] = rng.permutation(len(df))
df["random_coin"] = rng.integers(0, 2, len(df))
print(df.nunique().to_dict())
X_train, X_test, y_train, y_test = train_test_split(df, yc, test_size=0.3, random_state=0)
rfc = RandomForestClassifier(random_state=0, n_jobs=-1).fit(X_train, y_train)
print("train accuracy", rfc.score(X_train, y_train), " test accuracy", round(rfc.score(X_test, y_test), 3))

In [ ]:
perm = permutation_importance(rfc, X_test, y_test, n_repeats=20, random_state=0, n_jobs=-1)
pd.DataFrame({"impurity-based": rfc.feature_importances_, "permutation mean": perm.importances_mean,
              "permutation std": perm.importances_std}, index=df.columns).round(3)

In [ ]:
# permutation importance by hand for one column: shuffle it in the test set and measure the drop in accuracy
base = rfc.score(X_test, y_test)
shuffled = X_test.copy()
shuffled["x1"] = np.random.default_rng(1).permutation(shuffled["x1"].values)
print("accuracy", round(base, 3), "-> with x1 shuffled", round(rfc.score(shuffled, y_test), 3))